In [9]:
import sys
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.insert(0, os.getcwd())
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import joblib
from src.data_utils import load_config, load_raw_data, clean_data, save_processed
from src.features import encode_categoricals, scale_features, engineer_features, get_feature_matrix

config = load_config("config/config.yaml")
df_raw = load_raw_data(config)

Dataset cargado: 20718 filas, 27 columnas


In [10]:

df_clean = clean_data(df_raw, config)
print(f"\nShape final: {df_clean.shape}")

Duplicados eliminados: 82
Canciones virales (>= 150,000,000): 4687
viral
0    0.766386
1    0.233614
Name: proportion, dtype: float64

Shape final: (20063, 28)


In [11]:

df_feat = engineer_features(df_clean)
print("Features creadas:")
print([c for c in df_feat.columns if c not in df_raw.columns])

Features creadas:
['viral', 'engagement_rate', 'log_Views', 'log_Likes', 'log_Comments', 'log_Stream', 'popularity_score']


In [12]:
df_feat, encoders = encode_categoricals(df_feat, config["features_categoricas"])
joblib.dump(encoders, "models/label_encoders.pkl")
print(" Encoders guardados")

 Encoders guardados


In [13]:

X = get_feature_matrix(df_feat, config, use_log=True)
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)



In [14]:
y_clf  = df_feat[config["target_classification"]]

X_train, X_test, y_train_c, y_test_c = train_test_split(
    X, y_clf,
    test_size=config["test_size"],
    random_state=config["random_state"],
    stratify=y_clf
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Balance viral en train: {y_train_c.mean():.2%}")

Train: (16050, 14) | Test: (4013, 14)
Balance viral en train: 23.36%


In [15]:

X_train_sc, X_test_sc, scaler = scale_features(X_train, X_test)

In [16]:

import os; os.makedirs("../data/processed", exist_ok=True)
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

# Ruta absoluta
save_path = project_root / 'data' / 'processed' / 'splits.pkl'
save_path.parent.mkdir(parents=True, exist_ok=True)

# Guardar
joblib.dump((X_train, X_test, X_train_sc, X_test_sc,
             y_train_c, y_test_c,
             list(X.columns)), save_path)


print(df_feat["viral"].value_counts(normalize=True))

save_processed(df_feat, config)
print("\nPreparación completada. Datos listos para modelado.")

viral
0    0.766386
1    0.233614
Name: proportion, dtype: float64
 Datos guardados en data/processed/datos_limpios.csv

Preparación completada. Datos listos para modelado.
